In [31]:
import numpy as np
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import ast

from scipy.stats import linregress

from matplotlib.gridspec import GridSpec
from matplotlib.colors import hsv_to_rgb
import matplotlib.patches as patches

# Set font size for plots and font type to be Arial
plt.rcParams.update({'font.size': 12})
plt.rcParams['font.family'] = 'Arial'

### Extract rate

In [32]:
# data setting
Nm = 1250

epoch = 100
T, dt = 2500, 0.1

# range of parameters
recpar = 'rec_0_str'
# recpar = 'rec_30_str'
Nfs = [0, 25, 30, 50, 90, 150, 250]
Ws = [0.01, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5]
Bs = [0.1, 0.3, 0.5, 0.7, 0.9]

recgpe = './data/%s/gpe'%recpar
recmsn = './data/%s/msn'%recpar

exppath = './exp/%s'%recpar
pltpath = './plot/%s'%recpar

os.makedirs(exppath, exist_ok=True)
os.makedirs(pltpath, exist_ok=True)

flag_msn = False
flag_gpe = False

In [33]:
# extract msn stats
if flag_msn:
    # extract data from raw recordings
    frames = []
    binsin = 20
    binpop = 2
    for Nf in Nfs:
        if Nf > 0:
            nNums = [Nm, Nm, Nf]
        else:
            nNums = [Nm, Nm, 1]
        for W in Ws:
            for B in Bs:
                label = '/Nf%d-W%s-B%s/'%(Nf, W, B)
                data = np.load(recmsn + label + 'spk.npz')
                es, ts = data.f.arr_0, data.f.arr_1
                data.close()

                # processing data
                rates = []
                corrs = []
                rpops = []
                for e in range(epoch):
                    ta, tb = e*T+500, (e+1)*T
                    idx = (ts>=ta) & (ts<tb) & (es<=2*Nm)

                    idBins = np.arange(0, 2*Nm+1, 1)
                    tmBins = np.arange(0, T+1-500, binsin)
                    rate = np.histogram2d(ts[idx]-ta, es[idx], bins=[tmBins, idBins])[0]
                    rates.append([np.mean(rate[:, :Nm]), np.mean(rate[:, Nm:2*Nm])])

                    # correlations
                    coV = np.corrcoef(rate.T)
                    np.fill_diagonal(coV, 0.)
                    coV[np.isnan(coV)] = 0
                    ix = np.arange(0, Nm)
                    corrs.append([  np.sum(coV[np.ix_(ix, ix)])/Nm/(Nm-1), 
                                    np.sum(coV[np.ix_(ix+Nm, ix+Nm)])/Nm/(Nm-1),
                                    np.sum(coV[np.ix_(ix, ix+Nm)])/Nm/Nm])
                    
                    # population responses
                    idBins = [0.5, 2*Nm+0.5]
                    tmBins = np.arange(0, T+1-500, binpop)
                    rpops.append(np.histogram2d(ts[idx]-ta, es[idx], bins=[tmBins, idBins])[0])

                # output correlations
                corrs = np.array(corrs)
                rates = np.array(rates)*1e3/binsin
                
                # variability
                rpops = np.array(rpops).squeeze()*1e3/binpop/2/Nm
                fanos = rpops.var(axis=0) / rpops.mean(axis=0)
                fano = np.mean(fanos[~np.isnan(fanos)])

                frames.append([Nf, W, B, corrs[:,0].tolist(), corrs[:,1].tolist(), corrs[:,2].tolist(), rates[:,0].tolist(), rates[:,1].tolist(), fano])

    colname = ['Nf', 'Win', 'Bin', 'WoutA', 'WoutB', 'Bout', 'routA', 'routB', 'fano']
    stats = pd.DataFrame(frames, columns=colname)
    stats.to_csv(exppath + '/msn_stats.csv', index=False)

In [34]:
# extract gpe stats
if flag_gpe:
    isis_thres = 10  # ms
    nspk_thres = 3
    def cv2_from_isis(isis):
        """Compute CV2 for a 1D array of ISIs."""
        if len(isis) < 2:
            return np.nan  # not enough intervals

        I1 = isis[:-1]
        I2 = isis[1:]
        cv2_vals = 2.0 * np.abs(I2 - I1) / (I2 + I1)
        return np.mean(cv2_vals)

    frames = []
    spks = []
    for Nf in Nfs:
        for W in Ws:
            for B in Bs:
                label = '/Nf%d-W%s-B%s/'%(Nf, W, B)
                data = np.load(recgpe + label + 'spk.npz')
                ts = data.f.arr_0
                spks.append(ts)

                rs = []
                cvs = []
                cv2 = []
                bis = []
                for e in range(epoch):
                    ta, tb = e*T + 500, (e+1)*T
                    idx = (ts>=ta) & (ts<tb)
                    spike_times = ts[idx] - ta
                    total_spikes = len(spike_times)

                    if total_spikes == 0:
                        print("No spikes in epoch %d for Nf=%d, W=%s, B=%s"%(e, Nf, W, B))
                        rs.append(0.0)
                        cvs.append(np.nan)
                        cv2.append(np.nan)
                        bis.append(np.nan)
                        continue

                    # rate
                    rs.append(np.sum(idx) / ((T-500) * 1e-3))
                
                    # cv
                    isis = np.diff(spike_times)
                    cvs.append(np.std(isis) / np.mean(isis))
                    cv2.append(cv2_from_isis(isis))

                    # burst index (percentage of spike groups with ISI < isis_thres and nspikes >=3)
                    burst_spikes = 0
                    count = 0
                    for i in range(1, total_spikes):
                        if spike_times[i] - spike_times[i-1] < isis_thres:
                            count += 1
                        else:
                            if count >= nspk_thres - 1:
                                burst_spikes += count + 1
                            count = 0
                    bis.append(burst_spikes / total_spikes)

                # fano factor
                mean_rate = np.mean(rs)
                variance_rate = np.var(rs)
                if mean_rate == 0:
                    fano = np.nan  # Avoid division by zero
                else:
                    fano = variance_rate / mean_rate
                    
                frames.append([Nf, W, B, rs, cvs, cv2, fano, bis])
                
    # save to csv
    colname = ['Nf', 'Win', 'Bin', 'rout', 'cvout', 'cv2out', 'fanoout', 'biout']
    stats = pd.DataFrame(frames, columns=colname)
    stats.to_csv('./exp/%s/gpe_stats.csv'%recpar, index=False)

In [35]:
# combine msn and gpe stats
def parse_array_cell(x):
    # parses a cell that may contain a list/array in string form into a numpy array
    import re
    if pd.isna(x):
        return np.array([])
    if isinstance(x, (list, tuple, np.ndarray)):
        return np.asarray(x, dtype=float)
    s = str(x).strip()
    # try literal_eval
    try:
        val = ast.literal_eval(s)
        if isinstance(val, (list, tuple, np.ndarray)):
            return np.asarray(val, dtype=float)
        return np.asarray([float(val)], dtype=float)
    except Exception:
        pass
    # try eval with numpy context (e.g., 'np.array([1,2])')
    try:
        val = eval(s, {"np": np, "array": np.array})
        if isinstance(val, (list, tuple, np.ndarray)):
            return np.asarray(val, dtype=float)
        return np.asarray([float(val)], dtype=float)
    except Exception:
        pass
    # extract numeric tokens as last resort
    nums = re.findall(r"[-+]?\d*\.\d+|\d+", s)
    if nums:
        return np.asarray([float(n) for n in nums], dtype=float)
    return np.array([])

stats_msn = pd.read_csv('./exp/%s/msn_stats.csv'%recpar)
# convert WoutA/Bout/routA into arrays and store them
stats_msn['WoutA'] = stats_msn['WoutA'].apply(parse_array_cell)
stats_msn['WoutB'] = stats_msn['WoutB'].apply(parse_array_cell)
stats_msn['Bout'] = stats_msn['Bout'].apply(parse_array_cell)
stats_msn['routA'] = stats_msn['routA'].apply(parse_array_cell)
stats_msn['routB'] = stats_msn['routB'].apply(parse_array_cell)
# also keep mean values for easier plotting where desired
stats_msn['Wmsn'] = (stats_msn['WoutA'].apply(lambda a: float(np.mean(a)) if a.size>0 else np.nan) +
                    stats_msn['WoutB'].apply(lambda a: float(np.mean(a)) if a.size>0 else np.nan)) / 2.0
stats_msn['Bmsn'] = stats_msn['Bout'].apply(lambda a: float(np.mean(a)) if a.size>0 else np.nan)
stats_msn['rmsn'] = (stats_msn['routA'].apply(lambda a: float(np.mean(a)) if a.size>0 else np.nan) +
                    stats_msn['routB'].apply(lambda a: float(np.mean(a)) if a.size>0 else np.nan)) / 2.0
stats_msn['fanomsn'] = stats_msn['fano'].astype(float)

stats_gpe = pd.read_csv('./exp/%s/gpe_stats.csv'%recpar)
stats_gpe['rout'] = stats_gpe['rout'].apply(parse_array_cell)
stats_gpe['rgpe'] = stats_gpe['rout'].apply(lambda a: float(np.mean(a)) if a.size>0 else np.nan)
stats_gpe['cvout'] = stats_gpe['cvout'].apply(parse_array_cell)
stats_gpe['cvout'] = stats_gpe['cvout'].apply(lambda a: a[~np.isnan(a)])
stats_gpe['cvgpe'] = stats_gpe['cvout'].apply(lambda a: float(np.mean(a)) if a.size>0 else np.nan)
stats_gpe['cv2out'] = stats_gpe['cv2out'].apply(parse_array_cell)
stats_gpe['cv2out'] = stats_gpe['cv2out'].apply(lambda a: a[~np.isnan(a)])
stats_gpe['cv2gpe'] = stats_gpe['cv2out'].apply(lambda a: float(np.mean(a)) if a.size>0 else np.nan)
stats_gpe['biout'] = stats_gpe['biout'].apply(parse_array_cell)
stats_gpe['biout'] = stats_gpe['biout'].apply(lambda a: a[~np.isnan(a)])
stats_gpe['bigpe'] = stats_gpe['biout'].apply(lambda a: float(np.mean(a)) if a.size>0 else np.nan)
stats_gpe['fanogpe'] = stats_gpe['fanoout'].astype(float)

stats = pd.merge(stats_msn, stats_gpe, on=['Nf', 'Win', 'Bin'])
# set each entry of Win to a list with number of epoch (keep Win numeric for plotting)
stats['Win_arr'] = stats['Win'].apply(lambda v: np.full(epoch, float(v)))

# 2D color mapping: red intensity encodes Win, blue intensity encodes Bin
pairs_all = stats[['Win','Bin']].dropna().drop_duplicates()
wmin, wmax = pairs_all['Win'].min(), pairs_all['Win'].max()
bmin, bmax = pairs_all['Bin'].min(), pairs_all['Bin'].max()
def uv_to_color(u, v):
    # angle in [0,1]
    H = (np.arctan2(v, u) + np.pi) / (2*np.pi)
    # magnitude normalized to [0,1]
    S = np.sqrt(u**2 + v**2)
    S /= S.max() if S.max() != 0 else 1
    V = np.ones_like(H)
    return hsv_to_rgb(np.dstack((H, S, V)))

#### MSN activity

In [49]:
# Variability MSN
binsize = 10
tmBins  = np.arange(0, T+1-500, binsize)
idBins  = np.array([0, 2*Nm])
Nf_s    = [250, 25]
W_s     = [0.1, 0.1]
Bin = 0.9
colors  = ['blue', 'red']

fig = plt.figure(figsize=(8, 6))

# Outer GridSpec: separate upper (raster+rate) from lower (heatmaps)
gs_outer = GridSpec(2, 1, figure=fig, height_ratios=[1.2, 1], hspace=0.5)

# Inner GridSpec for upper two rows (raster + rate) with tight spacing
gs_upper = gs_outer[0].subgridspec(2, 4, 
                                    width_ratios=[1, 0.20, 1, 0.20],
                                    height_ratios=[1, 1],
                                    hspace=0.15, wspace=0.1)

# Inner GridSpec for bottom row (heatmaps)
gs_lower = gs_outer[1].subgridspec(1, 2, wspace=0.15)

# Axes (top row - rasters)
ax1   = fig.add_subplot(gs_upper[0, 0])
ax_n1 = fig.add_subplot(gs_upper[0, 1])
ax2   = fig.add_subplot(gs_upper[0, 2])
ax_n2 = fig.add_subplot(gs_upper[0, 3])

# Axes (middle row - rates)
ax3   = fig.add_subplot(gs_upper[1, 0])
ax_b1 = fig.add_subplot(gs_upper[1, 1])
ax4   = fig.add_subplot(gs_upper[1, 2])
ax_b2 = fig.add_subplot(gs_upper[1, 3])

# Axes (bottom row - heatmaps)
ax5 = fig.add_subplot(gs_lower[0, 0])
ax6 = fig.add_subplot(gs_lower[0, 1])

axs_raster = [ax1, ax2]
axs_rate   = [ax3, ax4]

# Add panel labels
ax1.text(-0.3, 1.4, 'A', transform=ax1.transAxes, fontsize=16, va='top')
ax2.text(-0.05, 1.4, 'B', transform=ax2.transAxes, fontsize=16, va='top')

# --- Main raster + rate plots ----------------------------------------
for i, (Nf, W) in enumerate(zip(Nf_s, W_s)):

    # Load
    label = f'/Nf{Nf}-W{W}-B{Bin}/'
    data  = np.load(recmsn + label + 'spk.npz')
    es, ts = data.f.arr_0, data.f.arr_1

    # ---------------- Raster plot (top row) ----------------
    ax = axs_raster[i]
    ax.set_title(rf'$N_{{fsi}}={Nf},\ W_{{in}}={W}$')

    for e, c in enumerate(colors):
        ta, tb = e*T, (e+1)*T
        idx = (ts >= ta) & (ts < tb)
        ax.scatter((ts[idx] - ta)/1000, es[idx], s=1, c=c)

    ax.set_xlim(1.4, 1.6)
    ax.set_ylim(0, 200)
    ax.set_xticks([])
    if i == 0:
        ax.set_ylabel('Nrn ID')
    else:
        ax.set_yticks([])

    # ---------------- Rate plot (bottom row) ----------------
    ax = axs_rate[i]

    for e, c in enumerate(colors):
        ta, tb = e*T + 500, (e+1)*T
        idx = (ts >= ta) & (ts < tb) & (es <= idBins[-1])
        rates = np.histogram2d(ts[idx] - ta, es[idx],
                               bins=[tmBins, idBins])[0]
        rates = rates * 1e3 / binsize / idBins[-1]
        ax.plot(tmBins[:-1] / 1000, rates, c=c, label=f'trial {e+1}')

    ax.set_xlim(1.4, 1.6)
    ax.set_ylim(0, 15)
    ax.set_xlabel('Time (s)')
    if i == 0:
        ax.legend()
        ax.set_ylabel(r'$r_{msn}$ (Hz)')
    else:
        ax.set_yticks([])

# --- Spontaneous activity small panels -------------------------------

# --- Nf = 25, W = -0.1 ---
Nf = 25; W = -0.1
label = f'/Nf{Nf}-W{W}-B{Bin}/'
data  = np.load(recmsn + label + 'spk.npz')
es, ts = data.f.arr_0, data.f.arr_1

# Small raster
ta, tb = e*T, (e+1)*T
idx = (ts >= ta) & (ts < tb)
ax_n1.scatter(ts[idx] - ta, es[idx], s=1, c='k')
ax_n1.set_xlim(950, 1050)
ax_n1.set_ylim(0, 200)
ax_n1.set_xticks([])
ax_n1.set_yticks([])
ax_n1.set_title('spon.')

# Small rate
ta2, tb2 = e*T + 500, (e+1)*T
idx = (ts >= ta2) & (ts < tb2) & (es <= idBins[-1])
rates = np.histogram2d(ts[idx] - ta2, es[idx],
                       bins=[tmBins, idBins])[0]
rates = rates * 1e3 / binsize / idBins[-1]
ax_b1.plot(tmBins[:-1], rates, c='k')
ax_b1.set_xlim(950, 1050)
ax_b1.set_ylim(0, 15)
ax_b1.set_xticks([])
ax_b1.set_yticks([])
ax_b1.set_xlabel('0.1s', labelpad=20)
print("Mean (Nf=25,W=-0.1):", np.mean(rates))

# --- Nf = 250, W = -0.1 ---
Nf = 250
label = f'/Nf{Nf}-W{W}-B{Bin}/'
data  = np.load(recmsn + label + 'spk.npz')
es, ts = data.f.arr_0, data.f.arr_1

# Small raster
ta, tb = e*T, (e+1)*T
idx = (ts >= ta) & (ts < tb)
ax_n2.scatter(ts[idx] - ta, es[idx], s=1, c='k')
ax_n2.set_xlim(950, 1050)
ax_n2.set_ylim(0, 200)
ax_n2.set_xticks([])
ax_n2.set_yticks([])
ax_n2.set_title('spon.')

# Small rate
ta2, tb2 = e*T + 500, (e+1)*T
idx = (ts >= ta2) & (ts < tb2) & (es <= idBins[-1])
rates = np.histogram2d(ts[idx] - ta2, es[idx],
                       bins=[tmBins, idBins])[0]
rates = rates * 1e3 / binsize / idBins[-1]
ax_b2.plot(tmBins[:-1], rates, c='k')
ax_b2.set_xlim(950, 1050)
ax_b2.set_ylim(0, 15)
ax_b2.set_xticks([])
ax_b2.set_yticks([])
ax_b2.set_xlabel('0.1s', labelpad=20)
print("Mean (Nf=250,W=-0.1):", np.mean(rates))

# --- Variability heatmaps --------------------------------------------
subset = stats[stats['Bin'] == Bin]

ax5.text(-0.5, 1.2, 'C', transform=ax5.transAxes, fontsize=16, va='top')
ax6.text(-0.15, 1.2, 'D', transform=ax6.transAxes, fontsize=16, va='top')

# Left heatmap - firing rate (move only Nf=0 row to bottom)
pivot_table = subset.pivot(index='Nf', columns='Win', values='rmsn')
nf0_row = pivot_table.loc[[0]]
pivot_table = pd.concat([pivot_table.drop(index=0), nf0_row])
sns.heatmap(pivot_table, annot=False, fmt=".2f", cmap='viridis', 
            cbar_kws={'label': r'$r_{msn}$ (Hz)'}, ax=ax5)

# --- Highlight the last row ---
nrows, ncols = pivot_table.shape
# Rectangle covering the last row (y = nrows-1, height = 1, width = ncols)
rect = patches.Rectangle(
    (0, nrows - 1),          # (x0, y0): start at first column, last row
    ncols, 1,                # width, height
    fill=False,
    edgecolor='crimson',
    linewidth=2,
    clip_on=False
)
ax5.add_patch(rect)

ax5.set_title('Mean firing rate')
ax5.set_xlabel(r'$W_{in}$ (FFE sharing)')
ax5.set_ylabel(r'$N_{fsi}$')

# extra y axis to denote FFI sharing level
par1 = ax5.secondary_yaxis(-0.3)
par1.yaxis.set_label_position('left')
par1.yaxis.set_ticks_position('left')
par1.set_ylabel('FFI sharing')

# --- Shorten the secondary y-axis to exclude the last row ---
nrows = pivot_table.shape[0]
y0, y1 = ax5.get_ylim()

# Match the direction of the primary axis but stop at the top edge of the last row
if y0 > y1:
    # Inverted axis (common with heatmaps): e.g., (nrows, 0)
    par1.set_ylim(y0 - 1, y1)   # cut off 1 unit from the top (last row height)
else:
    # Normal direction
    par1.set_ylim(y0, y1 - 1)

# Place ticks only for the first nrows-1 row centers
yticks = np.arange(nrows - 1) + 0.5
# Your labels (excluding the last row), using Nfs[1:] as in your code
ylabels = [f'{15/y:0.2f}' for y in Nfs[1:]]  # assumes len(Nfs[1:]) == nrows-1
par1.set_yticks(yticks, labels=ylabels)

# Optional: also shorten the visible spine length for a clean cutoff
par1.spines['left'].set_bounds(0, nrows - 1)

# Right heatmap - Fano factor
pivot_table = subset.pivot(index='Nf', columns='Win', values='fano')
nf0_row = pivot_table.loc[[0]]
pivot_table = pd.concat([pivot_table.drop(index=0), nf0_row])
sns.heatmap(pivot_table, annot=False, fmt=".2f", cmap='cividis', 
            cbar_kws={'label': r'$FF_{msn}$'}, ax=ax6)

# --- Highlight the last row ---
nrows, ncols = pivot_table.shape
# Rectangle covering the last row (y = nrows-1, height = 1, width = ncols)
rect = patches.Rectangle(
    (0, nrows - 1),          # (x0, y0): start at first column, last row
    ncols, 1,                # width, height
    fill=False,
    edgecolor='crimson',
    linewidth=2,
    clip_on=False
)
ax6.add_patch(rect)

ax6.set_title('Across-trial variability')
ax6.axis('off')

# --- Final layout ----------------------------------------------------
plt.suptitle(r'MSN Population Across-trial Variability ($B_{in}$' + f'={Bin})', y=0.995)
plt.savefig(pltpath + '/var_msn.eps', dpi=300, bbox_inches='tight')
plt.close()

Mean (Nf=25,W=-0.1): 0.9743999999999999
Mean (Nf=250,W=-0.1): 1.0051999999999999


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


In [37]:
# Correlation transfer - Combined figure
nfs = [250, 150, 50, 25]
ncols = len(nfs)

# Create combined figure
fig = plt.figure(figsize=(10, 9))

# Split into two units: row 1 (top) and rows 2-3 (bottom group)
gs_outer = GridSpec(2, 1, figure=fig, height_ratios=[1, 2], hspace=0.35)

# Row 1: Split into two groups - A separate, and [B, C] together with reduced spacing
gs_row1 = gs_outer[0].subgridspec(1, 2, width_ratios=[1, 2], wspace=0.3)
ax_in = fig.add_subplot(gs_row1[0, 0])
gs_row1_right = gs_row1[0, 1].subgridspec(1, 2, wspace=0.2)
ax_out = fig.add_subplot(gs_row1_right[0, 0])
ax_scatter = fig.add_subplot(gs_row1_right[0, 1])

# Rows 2 and 3 as a single unit with reduced spacing between them
gs_bottom = gs_outer[1].subgridspec(2, 1, height_ratios=[1, 1], hspace=0.5)

# Row 2: cortran_base0 (4 plots)
gs_row2 = gs_bottom[0].subgridspec(1, ncols, wspace=0.2)
axes_250 = [fig.add_subplot(gs_row2[0, i]) for i in range(ncols)]

# Row 3: cortran_base250 (4 plots)
gs_row3 = gs_bottom[1].subgridspec(1, ncols, wspace=0.2)
axes_0 = [fig.add_subplot(gs_row3[0, i]) for i in range(ncols)]

# --- Row 1, Panel A: Input correlation ---
baseNf = 0
for _, row in pairs_all.iterrows():
    c = uv_to_color(row.Win, row.Bin)
    ax_in.scatter(row.Win, np.multiply(row.Win, row.Bin), s=30, color=c, edgecolor='none')
ax_in.set_title('Input correlation')
ax_in.set_xlabel(r'$W_{in}$')
ax_in.set_ylabel(r'$B_{in}xW_{in}$')
ax_in.text(-0.15, 1.3, 'A', transform=ax_in.transAxes, fontsize=16, va='top')

# --- Row 1, Panel B: Output correlation ---
for _, row in pairs_all.iterrows():
    c = uv_to_color(row.Win, row.Bin)
    sub = stats[(stats['Win']==row.Win) & (stats['Bin']==row.Bin) & (stats['Nf']==baseNf)]
    if not sub.empty:
        wmsn = sub['Wmsn'].values[0]
        bmsn = sub['Bmsn'].values[0]
        ax_out.scatter(wmsn, bmsn, s=30, color=c, edgecolor='none')
ax_out.set_title('Output correlation')
ax_out.set_xlabel(r'$W_{msn}$')
ax_out.set_ylabel(r'$B_{msn}$')
ax_out.set_xlim([-0.01, 0.09])
ax_out.set_ylim([-0.01, 0.09])
ax_out.text(-0.15, 1.3, 'B', transform=ax_out.transAxes, fontsize=16, va='top')

Wouts = (stats['WoutA'] + stats['WoutB'])/2
Bouts = stats['Bout']
for i in range(len(stats)):
    Wout = Wouts.iloc[i]
    Bout = Bouts.iloc[i]
    c = uv_to_color(stats['Win'].iloc[i], stats['Bin'].iloc[i])
    ax_scatter.scatter(Wout, Bout, color=c, s=1, alpha=0.7, edgecolor='none')
ax_scatter.set_title('All trials')
ax_scatter.set_xlim([-0.01, 0.09])
ax_scatter.set_ylim([-0.01, 0.09])
ax_scatter.tick_params(labelleft=False, labelbottom=False)
ax_scatter.grid(True, alpha=0.5)

# ax_scatter.text(-0.15, 1.3, 'C', transform=ax_scatter.transAxes, fontsize=16, va='top')

# --- Row 2: Displacements from Nf=0 ---
baseNf = 0
base_wb = stats.loc[stats['Nf']==baseNf, ['Win','Bin','Wmsn','Bmsn']].copy().replace([np.inf,-np.inf], np.nan).dropna(subset=['Wmsn','Bmsn'])
all_wb = stats[['Wmsn','Bmsn']].replace([np.inf,-np.inf], np.nan).dropna()
wxmin, wxmax = all_wb['Wmsn'].min(), all_wb['Wmsn'].max()
bymin, bymax = all_wb['Bmsn'].min(), all_wb['Bmsn'].max()
wpx = 0.05*(wxmax-wxmin if wxmax>wxmin else 1.0); bpy = 0.05*(bymax-bymin if bymax>bymin else 1.0)
wxmin -= wpx; wxmax += wpx; bymin -= bpy; bymax += bpy

for idx, nf in enumerate(nfs):
    ax = axes_250[idx]
    cur_wb = stats.loc[stats['Nf']==nf, ['Win','Bin','Wmsn','Bmsn']].copy().replace([np.inf,-np.inf], np.nan).dropna(subset=['Wmsn','Bmsn'])
    pair_wb = base_wb.merge(cur_wb, on=['Win','Bin'], suffixes=('_base','_cur'), how='inner')
    for _, row in pair_wb.iterrows():
        c = uv_to_color(row.Win, row.Bin)
        # ax.scatter(row['Wmsn_base'], row['Bmsn_base'], s=22, color=c, edgecolor='none')
        ax.scatter(row['Wmsn_cur'], row['Bmsn_cur'], s=22, color=c, edgecolor='none')
        ax.quiver(row['Wmsn_base'], row['Bmsn_base'], row['Wmsn_cur']-row['Wmsn_base'], row['Bmsn_cur']-row['Bmsn_base'],
                    angles='xy', scale_units='xy', scale=1.0, width=0.003, color=c, alpha=0.85)
    ax.set_xlim(wxmin,wxmax)
    ax.set_ylim(bymin,bymax)
    ax.set_title(r'$N_{fsi}$' + f'={nf}')
    if idx == 0:
        ax.set_xlabel(r'$W_{msn}$')
        ax.set_ylabel(r'$B_{msn}$')
        ax.text(-0.25, 1.3, 'C', transform=ax.transAxes, fontsize=16, va='top')
    else:
        ax.tick_params(labelleft=False, labelbottom=False)
        ax.grid(True, alpha=0.5)

# --- Row 3: Displacements from Nf=250 ---
baseNf = 250
base_wb = stats.loc[stats['Nf']==baseNf, ['Win','Bin', 'Wmsn','Bmsn']].copy().replace([np.inf,-np.inf], np.nan).dropna(subset=['Wmsn','Bmsn'])
all_wb = stats[['Wmsn','Bmsn']].replace([np.inf,-np.inf], np.nan).dropna()
wxmin, wxmax = all_wb['Wmsn'].min(), all_wb['Wmsn'].max()
bymin, bymax = all_wb['Bmsn'].min(), all_wb['Bmsn'].max()
wpx = 0.05*(wxmax-wxmin if wxmax>wxmin else 1.0); bpy = 0.05*(bymax-bymin if bymax>bymin else 1.0)
wxmin -= wpx; wxmax += wpx; bymin -= bpy; bymax += bpy

for idx, nf in enumerate(nfs):
    ax = axes_0[idx]
    cur_wb = stats.loc[stats['Nf']==nf, ['Win','Bin', 'Wmsn','Bmsn']].copy().replace([np.inf,-np.inf], np.nan).dropna(subset=['Wmsn','Bmsn'])
    pair_wb = base_wb.merge(cur_wb, on=['Win','Bin'], suffixes=('_base','_cur'), how='inner')
    for _, row in pair_wb.iterrows():
        c = uv_to_color(row.Win, row.Bin)
        # ax.scatter(row['Wmsn_base'], row['Bmsn_base'], s=22, color=c, edgecolor='none')
        ax.scatter(row['Wmsn_cur'], row['Bmsn_cur'], s=22, color=c, edgecolor='none')
        ax.quiver(row['Wmsn_base'], row['Bmsn_base'], row['Wmsn_cur']-row['Wmsn_base'], row['Bmsn_cur']-row['Bmsn_base'],
                    angles='xy', scale_units='xy', scale=1.0, width=0.003, color=c, alpha=0.85)
    ax.set_xlim(wxmin,wxmax)
    ax.set_ylim(bymin,bymax)
    if idx == 0:
        ax.set_xlabel(r'$W_{msn}$')
        ax.set_ylabel(r'$B_{msn}$')
        ax.text(-0.25, 1.3, 'C', transform=ax.transAxes, fontsize=16, va='top')
    else:
        ax.tick_params(labelleft=False, labelbottom=False)
        ax.grid(True, alpha=0.5)

plt.suptitle('Correlation Transfer in MSN Population', y=0.995, fontsize=14)
plt.savefig(pltpath + '/cor_msn.eps', dpi=300, bbox_inches='tight')
plt.close()

The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


#### GPe activity

In [51]:
# -------------------------
# TOP FIGURE: MSN & GPe transfer/quiver
# -------------------------
baseNf = 250
nfs = [250, 150, 50, 25]
ncols = len(nfs)

fig_top = plt.figure(figsize=(10, 6))
gs_top_outer = GridSpec(2, 1, figure=fig_top, height_ratios=[1, 1], hspace=0.6)

# Row 1: MSN panels
gs_row1_msn = gs_top_outer[0].subgridspec(1, ncols, wspace=0.1)
axes_msn = [fig_top.add_subplot(gs_row1_msn[0, i]) for i in range(ncols)]

# Row 2: GPe panels
gs_row2_gpe = gs_top_outer[1].subgridspec(1, ncols, wspace=0.1)
axes_gpe = [fig_top.add_subplot(gs_row2_gpe[0, i]) for i in range(ncols)]

# Panel labels
axes_msn[0].text(-0.45, 1.25, 'A', transform=axes_msn[0].transAxes, fontsize=16, va='top')
axes_gpe[0].text(-0.45, 1.25, 'B', transform=axes_gpe[0].transAxes, fontsize=16, va='top')

# ----- MSN transfer (upper row) -----
base_wb_msn = (
    stats.loc[stats['Nf'] == baseNf, ['Win', 'Bin', 'Wmsn', 'fanomsn']]
    .copy().replace([np.inf, -np.inf], np.nan).dropna(subset=['Wmsn', 'fanomsn'])
)
all_wb_msn = (
    stats[['Wmsn', 'fanomsn']]
    .replace([np.inf, -np.inf], np.nan).dropna()
)
wxmin, wxmax = all_wb_msn['Wmsn'].min(), all_wb_msn['Wmsn'].max()
bymin, bymax = all_wb_msn['fanomsn'].min(), all_wb_msn['fanomsn'].max()
wpx = 0.05 * (wxmax - wxmin if wxmax > wxmin else 1.0)
bpy = 0.05 * (bymax - bymin if bymax > bymin else 1.0)
wxmin -= wpx; wxmax += wpx; bymin -= bpy; bymax += bpy

for idx, nf in enumerate(nfs):
    ax = axes_msn[idx]
    cur_wb = (
        stats.loc[stats['Nf'] == nf, ['Win', 'Bin', 'Wmsn', 'fanomsn']]
        .copy().replace([np.inf, -np.inf], np.nan).dropna(subset=['Wmsn', 'fanomsn'])
    )
    pair_wb = base_wb_msn.merge(cur_wb, on=['Win', 'Bin'], suffixes=('_base', '_cur'), how='inner')
    for _, row in pair_wb.iterrows():
        c = uv_to_color(row.Win, row.Bin)
        ax.scatter(row['Wmsn_base'], row['fanomsn_base'], s=22, color=c, edgecolor='none')
        ax.quiver(
            row['Wmsn_base'], row['fanomsn_base'],
            row['Wmsn_cur'] - row['Wmsn_base'], row['fanomsn_cur'] - row['fanomsn_base'],
            angles='xy', scale_units='xy', scale=1.0, width=0.003, color=c, alpha=0.85
        )
    ax.set_title(r'$N_{fsi}$' + f'={nf}')
    ax.set_xlim(wxmin, wxmax)
    ax.set_ylim(bymin, bymax)
    if idx == 0:
        ax.set_xlabel(r'$W_{msn}$')
        ax.set_ylabel(r'$FF_{msn}$')
    else:
        ax.tick_params(labelleft=False, labelbottom=False)
    ax.grid(True, alpha=0.5)

# ----- GPe transfer (second row) -----
base_wb_gpe = (
    stats.loc[stats['Nf'] == baseNf, ['Win', 'Bin', 'fanogpe', 'bigpe']]
    .copy().replace([np.inf, -np.inf], np.nan).dropna(subset=['fanogpe', 'bigpe'])
)
all_wb_gpe = (
    stats[['fanogpe', 'bigpe']]
    .replace([np.inf, -np.inf], np.nan).dropna()
)
wxmin, wxmax = all_wb_gpe['bigpe'].min(), all_wb_gpe['bigpe'].max()
bymin, bymax = all_wb_gpe['fanogpe'].min(), all_wb_gpe['fanogpe'].max()
wpx = 0.05 * (wxmax - wxmin if wxmax > wxmin else 1.0)
bpy = 0.05 * (bymax - bymin if bymax > bymin else 1.0)
wxmin -= wpx; wxmax += wpx; bymin -= bpy; bymax += bpy

for idx, nf in enumerate(nfs):
    ax = axes_gpe[idx]
    cur_wb = (
        stats.loc[stats['Nf'] == nf, ['Win', 'Bin', 'fanogpe', 'bigpe']]
        .copy().replace([np.inf, -np.inf], np.nan).dropna(subset=['fanogpe', 'bigpe'])
    )
    pair_wb = base_wb_gpe.merge(cur_wb, on=['Win', 'Bin'], suffixes=('_base', '_cur'), how='inner')
    for _, row in pair_wb.iterrows():
        c = uv_to_color(row.Win, row.Bin)
        ax.scatter(row['bigpe_base'], row['fanogpe_base'], s=22, color=c, edgecolor='none')
        ax.quiver(
            row['bigpe_base'], row['fanogpe_base'],
            row['bigpe_cur'] - row['bigpe_base'], row['fanogpe_cur'] - row['fanogpe_base'],
            angles='xy', scale_units='xy', scale=1.0, width=0.003, color=c, alpha=0.85
        )
    ax.set_xlim(wxmin, wxmax)
    ax.set_ylim(bymin, bymax)
    if idx == 0:
        ax.set_xlabel(r'$BI_{gpe}$')
        ax.set_ylabel(r'$FF_{gpe}$')
    else:
        ax.tick_params(labelleft=False, labelbottom=False)
        ax.tick_params(bottom=False, left=False)
    ax.grid(True, alpha=0.5)

# Save top figure
fig_top.savefig(pltpath + '/transgpe.eps', dpi=300)
plt.close(fig_top)


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


In [50]:
# -------------------------
# BOTTOM FIGURE: rgpe distributions & GPe raster/ISI
# -------------------------
fig_bot = plt.figure(figsize=(10, 6))
gs_bot_outer = GridSpec(2, 1, figure=fig_bot, height_ratios=[1, 1], hspace=0.6)

# Row 3: Distribution of rgpe across trials (3 panels)
gs_row3_rgpe = gs_bot_outer[0].subgridspec(1, 3, wspace=0.2)
axes_rgpe = [fig_bot.add_subplot(gs_row3_rgpe[0, i]) for i in range(3)]
axes_rgpe[0].text(-0.35, 1.25, 'A', transform=axes_rgpe[0].transAxes, fontsize=16, va='top')

# Row 4: Raster (2 left panels) + ISI KDE (right panel)
gs_row4 = gs_bot_outer[1].subgridspec(1, 2, width_ratios=[2, 1], wspace=0.3)
gs_row4_left = gs_row4[0, 0].subgridspec(1, 2, wspace=0.1)
axes_fgpe = [
    fig_bot.add_subplot(gs_row4_left[0, 0]),
    fig_bot.add_subplot(gs_row4_left[0, 1]),
    fig_bot.add_subplot(gs_row4[0, 1]),
]
axes_fgpe[0].text(-0.40, 1.25, 'B', transform=axes_fgpe[0].transAxes, fontsize=16, va='top')

# ----- rgpe distributions -----
samples = [(0.01, 0.5), (0.01, 0.9), (0.45, 0.3)]
cs = ['blue', 'orange']

for i, (W, B) in enumerate(samples):
    subset = stats[(stats['Win'] == W) & (stats['Bin'] == B)]
    msn_subset = stats_msn[(stats_msn['Win'] == W) & (stats_msn['Bin'] == B)]
    ax = axes_rgpe[i]
    ax.set_title(r'$W_{in}$' + f'={W} ' + r'$B_{in}$' + f'={B}')
    for nf, c in zip([25, 250], cs):
        cur = subset[subset['Nf'] == nf]
        rgpe_values = cur['rout'].values[0]

        cur_msn = msn_subset[msn_subset['Nf'] == nf]
        rmsn_values = cur_msn['routA'].values[0]
        mean_rmsn = np.mean(rmsn_values)  # (not displayed but kept if needed)

        sns.kdeplot(rgpe_values, label=r'$N_{fsi}$' + f'={nf}', fill=True, alpha=0.5, color=c, ax=ax)
        ax.axvline(np.mean(rgpe_values), linestyle='--', color=c)
    if i == 0:
        ax.legend()
        ax.set_xlabel('rgpe (Hz)')
        ax.set_ylabel('Density')
    else:
        ax.tick_params(labelleft=False, labelbottom=False)
        ax.tick_params(bottom=False, left=False)
        ax.set(ylabel='')
    ax.set_xlim([15, 65])

# ----- GPe raster & ISI KDE -----
W, B = 0.01, 0.5
Nf_1, Nf_2 = 25, 250

# Load spike times
label1 = f'/Nf{Nf_1}-W{W}-B{B}/'
data1 = np.load(recgpe + label1 + 'spk.npz')
ts_1 = data1.f.arr_0
isi_1 = np.diff(ts_1)

spikes_1 = []
for e in range(epoch):
    ta, tb = e * T, (e + 1) * T
    idx = (ts_1 >= ta) & (ts_1 < tb)
    spikes_1.append(ts_1[idx] - e * T)

label2 = f'/Nf{Nf_2}-W{W}-B{B}/'
data2 = np.load(recgpe + label2 + 'spk.npz')
ts_2 = data2.f.arr_0
isi_2 = np.diff(ts_2)

spikes_2 = []
for e in range(epoch):
    ta, tb = e * T, (e + 1) * T
    idx = (ts_2 >= ta) & (ts_2 < tb)
    spikes_2.append(ts_2[idx] - e * T)

# Axes
ax1, ax2, ax3 = axes_fgpe

# Raster 1
for e, st in enumerate(spikes_1):
    y = np.full_like(st, e)
    ax1.scatter(st, y, s=1, color='black')
ax1.set_xlim(1000, 1500)
ax1.set_ylim(-1, 10)
ax1.set_xlabel('Time (ms)')
ax1.set_ylabel('Epoch')
ax1.set_title(r'$N_{fsi}$' + f'={Nf_1}')

# Raster 2
for e, st in enumerate(spikes_2):
    y = np.full_like(st, e)
    ax2.scatter(st, y, s=1, color='black')
ax2.set_xlim(1000, 1500)
ax2.set_ylim(-1, 10)
ax2.tick_params(labelleft=False, labelbottom=False)
ax2.tick_params(bottom=False, left=False)
ax2.set_title(r'$N_{fsi}$' + f'={Nf_2}')

# ISI KDE
sns.kdeplot(isi_1, bw_adjust=0.5, lw=2, label=r'$N_{fsi}$' + f'={Nf_1}', ax=ax3)
ax3.axvline(np.mean(isi_1), lw=1, ls='--', color='b')
sns.kdeplot(isi_2, bw_adjust=0.5, lw=2, label=r'$N_{fsi}$' + f'={Nf_2}', ax=ax3)
ax3.axvline(np.mean(isi_2), lw=1, ls='--', color='orange')
ax3.set_xlim([0, 40])
ax3.set_xlabel('Inter-spike Interval (ms)')
ax3.set_ylabel('Density')
ax3.set_title('ISI Distribution')
ax3.legend()
ax3.grid(True, alpha=0.3)

# Save bottom figure
fig_bot.savefig(pltpath + '/gpespk.eps', dpi=300)
plt.close(fig_bot)


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


#### Extra stuff

In [48]:
# Combined output-rate heatmaps (row 1), Wmsn heatmaps (row 2), and rate-diff scatters (row 3)
Nf_1, Nf_2 = 25, 250

subset = stats[stats['Nf'] == Nf_1]
heatmap_25 = subset.pivot(index='Bin', columns='Win', values='rmsn')

subset = stats[stats['Nf'] == Nf_2]
heatmap_250 = subset.pivot(index='Bin', columns='Win', values='rmsn')

heatmap_diff = heatmap_25 - heatmap_250

# Rate and correlation differences
subset_1 = stats_msn[stats_msn['Nf'] == Nf_1]
subset_2 = stats_msn[stats_msn['Nf'] == Nf_2]

Wmsn_1 = subset_1.pivot(index='Bin', columns='Win', values='Wmsn')
Bmsn_1 = subset_1.pivot(index='Bin', columns='Win', values='Bmsn')
rmsn_1 = subset_1.pivot(index='Bin', columns='Win', values='rmsn')

Wmsn_2 = subset_2.pivot(index='Bin', columns='Win', values='Wmsn')
Bmsn_2 = subset_2.pivot(index='Bin', columns='Win', values='Bmsn')
rmsn_2 = subset_2.pivot(index='Bin', columns='Win', values='rmsn')

Wmsn_diff = Wmsn_1 - Wmsn_2
Bmsn_diff = Bmsn_1 - Bmsn_2
rmsn_diff = rmsn_1 - rmsn_2

def scatter_diff(ax, xvals, yvals, xlabel, ylabel, title):
    x = xvals.values.flatten(); y = yvals.values.flatten()
    m = np.isfinite(x) & np.isfinite(y)
    ax.scatter(x[m], y[m], s=20, alpha=0.7)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(True, alpha=0.4)

# Common color scale for rate heatmaps
rate_min = np.nanmin(np.concatenate([heatmap_25.values.flatten(), heatmap_250.values.flatten()]))
rate_max = np.nanmax(np.concatenate([heatmap_25.values.flatten(), heatmap_250.values.flatten()]))
# Common color scale for Wmsn heatmaps
wmsn_min = np.nanmin(np.concatenate([Wmsn_1.values.flatten(), Wmsn_2.values.flatten()]))
wmsn_max = np.nanmax(np.concatenate([Wmsn_1.values.flatten(), Wmsn_2.values.flatten()]))
# Common color scale for Bmsn heatmaps
bmsn_min = np.nanmin(np.concatenate([Bmsn_1.values.flatten(), Bmsn_2.values.flatten()]))
bmsn_max = np.nanmax(np.concatenate([Bmsn_1.values.flatten(), Bmsn_2.values.flatten()]))

fig = plt.figure(figsize=(12, 10))
gs_outer = GridSpec(4, 1, figure=fig, height_ratios=[1, 1, 1, 1.5], hspace=0.45)
# widen the gap so colorbars don't overlap the last heatmap column
gs_top = gs_outer[0].subgridspec(1, 2, wspace=0.2, width_ratios=[2, 1])
gs_mid = gs_outer[1].subgridspec(1, 2, wspace=0.2, width_ratios=[2, 1])
gs_dow = gs_outer[2].subgridspec(1, 2, wspace=0.2, width_ratios=[2, 1])
gs_bottom = gs_outer[3].subgridspec(1, 3, wspace=0.2, width_ratios=[0.1, 1, 1])

# Row 1: rmsn heatmaps + colorbars
ax_hleft, ax_hright = gs_top[0].subgridspec(1, 3, width_ratios=[1, 1, 0.05]), gs_top[1].subgridspec(1, 2, width_ratios=[1, 0.05])
ax_h1, ax_h2, cb_rate = fig.add_subplot(ax_hleft[0, 0]), fig.add_subplot(ax_hleft[0, 1]), fig.add_subplot(ax_hleft[0, 2])
ax_h3, cb_rate_diff = fig.add_subplot(ax_hright[0, 0]), fig.add_subplot(ax_hright[0, 1])
ax_h1.text(-0.2, 1.2, 'A', transform=ax_h1.transAxes, fontsize=16, va='top')
im_h1 = sns.heatmap(heatmap_25, annot=False, fmt=".2f", cmap='viridis', vmin=rate_min, vmax=rate_max, cbar=False, ax=ax_h1)
ax_h1.set_title(r'$N_{fsi}$' + f'={Nf_1}')
im_h2 = sns.heatmap(heatmap_250, annot=False, fmt=".2f", cmap='viridis', vmin=rate_min, vmax=rate_max, cbar=False, ax=ax_h2)
ax_h2.set_title( r'$N_{fsi}$' + f'={Nf_2}')
im_h3 = sns.heatmap(heatmap_diff, annot=False, fmt=".2f", cmap='coolwarm', cbar=False, ax=ax_h3)
ax_h3.set_title('Difference')
fig.colorbar(im_h1.collections[0], cax=cb_rate, label=r'$r_{msn}$ (Hz)')
fig.colorbar(im_h3.collections[0], cax=cb_rate_diff, label=r'$\Delta r_{msn}$ (Hz)')
for ax in [ax_h1]:
    ax.set_xlabel(r'$W_{in}$'); ax.set_ylabel(r'$B_{in}$')
for ax in [ax_h2, ax_h3]:
    ax.set_xlabel(''); ax.set_ylabel(''); ax.set_xticks([]); ax.set_yticks([])

# Row 2: wmsn heatmaps + colorbars
ax_wleft, ax_wright = gs_mid[0].subgridspec(1, 3, width_ratios=[1, 1, 0.05]), gs_mid[1].subgridspec(1, 2, width_ratios=[1, 0.05])
ax_w1, ax_w2, cb_w = fig.add_subplot(ax_wleft[0, 0]), fig.add_subplot(ax_wleft[0, 1]), fig.add_subplot(ax_wleft[0, 2])
ax_w3, cb_w_diff = fig.add_subplot(ax_wright[0, 0]), fig.add_subplot(ax_wright[0, 1])
ax_w1.text(-0.2, 1.2, 'B', transform=ax_w1.transAxes, fontsize=16, va='top')
im_w1 = sns.heatmap(Wmsn_1, annot=False, fmt=".3f", cmap='viridis', vmin=wmsn_min, vmax=wmsn_max, cbar=False, ax=ax_w1)
im_w2 = sns.heatmap(Wmsn_2, annot=False, fmt=".3f", cmap='viridis', vmin=wmsn_min, vmax=wmsn_max, cbar=False, ax=ax_w2)
im_w3 = sns.heatmap(Wmsn_diff, annot=False, fmt=".3f", cmap='coolwarm', cbar=False, ax=ax_w3)
fig.colorbar(im_w1.collections[0], cax=cb_w, label=r'$W_{msn}$')
fig.colorbar(im_w3.collections[0], cax=cb_w_diff, label=r'$\Delta W_{msn}$')
for ax in [ax_w1, ax_w2, ax_w3]:
    ax.set_xlabel(''); ax.set_ylabel(''); ax.set_xticks([]); ax.set_yticks([])

# Row 3: bmsn heatmaps + colorbars
ax_bleft, ax_bright = gs_dow[0].subgridspec(1, 3, width_ratios=[1, 1, 0.05]), gs_dow[1].subgridspec(1, 2, width_ratios=[1, 0.05])
ax_b1, ax_b2, cb_b = fig.add_subplot(ax_bleft[0, 0]), fig.add_subplot(ax_bleft[0, 1]), fig.add_subplot(ax_bleft[0, 2])
ax_b3, cb_b_diff = fig.add_subplot(ax_bright[0, 0]), fig.add_subplot(ax_bright[0, 1])
ax_b1.text(-0.2, 1.2, 'C', transform=ax_b1.transAxes, fontsize=16, va='top')
im_b1 = sns.heatmap(Bmsn_1, annot=False, fmt=".3f", cmap='viridis', vmin=bmsn_min, vmax=bmsn_max, cbar=False, ax=ax_b1)
im_b2 = sns.heatmap(Bmsn_2, annot=False, fmt=".3f", cmap='viridis', vmin=bmsn_min, vmax=bmsn_max, cbar=False, ax=ax_b2)
im_b3 = sns.heatmap(Bmsn_diff, annot=False, fmt=".3f", cmap='coolwarm', cbar=False, ax=ax_b3)
fig.colorbar(im_b1.collections[0], cax=cb_b, label=r'$B_{msn}$')
fig.colorbar(im_b3.collections[0], cax=cb_b_diff, label=r'$\Delta B_{msn}$')
for ax in [ax_b1, ax_b2, ax_b3]:
    ax.set_xlabel(''); ax.set_ylabel(''); ax.set_xticks([]); ax.set_yticks([])

# Row 4: scatters (two columns)
ax_s0 = fig.add_subplot(gs_bottom[0, 0])
ax_s0.axis('off')
ax_s1 = fig.add_subplot(gs_bottom[0, 1])
ax_s2 = fig.add_subplot(gs_bottom[0, 2], sharex=ax_s1, sharey=ax_s1)
ax_s0.text(-0.2, 1.2, 'D', transform=ax_s0.transAxes, fontsize=16, va='top')
scatter_diff(ax_s1, rmsn_diff, Wmsn_diff,  r'$\Delta r_{msn}$', r'$\Delta W_{msn}$', 'Within-group correlation')
x = rmsn_diff.values.flatten(); y = Wmsn_diff.values.flatten()
m = np.isfinite(x) & np.isfinite(y)
slope, intercept, r_value, p_value, std_err = linregress(x, y)
print(r_value, p_value)
scatter_diff(ax_s2, rmsn_diff, Bmsn_diff,  '', r'$\Delta B_{msn}$', 'Between-group correlation')
x = rmsn_diff.values.flatten(); y = Bmsn_diff.values.flatten()
m = np.isfinite(x) & np.isfinite(y)
slope, intercept, r_value, p_value, std_err = linregress(x, y)
print(r_value, p_value)
plt.setp(ax_s2.get_yticklabels(), visible=False)
plt.setp(ax_s2.get_xticklabels(), visible=False)

plt.savefig(pltpath + '/rc_msn.eps', dpi=300, bbox_inches='tight')
plt.close()

-0.014718698289290578 0.9150624418543554
0.1571562382790704 0.2518493238908262


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


In [40]:
# Combined output-rate heatmaps (row 1), FFgpe heatmaps (row 2), and rate-diff scatters (row 3)
Nf_1, Nf_2 = 25, 250

# Rate and correlation differences
subset_1 = stats_gpe[stats_gpe['Nf'] == Nf_1]
subset_2 = stats_gpe[stats_gpe['Nf'] == Nf_2]

FF_1 = subset_1.pivot(index='Bin', columns='Win', values='fanogpe')
BI_1 = subset_1.pivot(index='Bin', columns='Win', values='bigpe')
rgpe_1 = subset_1.pivot(index='Bin', columns='Win', values='rgpe')

FF_2 = subset_2.pivot(index='Bin', columns='Win', values='fanogpe')
BI_2 = subset_2.pivot(index='Bin', columns='Win', values='bigpe')
rgpe_2 = subset_2.pivot(index='Bin', columns='Win', values='rgpe')

FF_diff = FF_1 - FF_2
BI_diff = BI_1 - BI_2
rgpe_diff = rgpe_1 - rgpe_2

def scatter_diff(ax, xvals, yvals, xlabel, ylabel, title):
    x = xvals.values.flatten(); y = yvals.values.flatten()
    m = np.isfinite(x) & np.isfinite(y)
    ax.scatter(x[m], y[m], s=20, alpha=0.7)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(True, alpha=0.4)

# Common color scale for rate heatmaps
rate_min = np.nanmin(np.concatenate([rgpe_1.values.flatten(), rgpe_2.values.flatten()]))
rate_max = np.nanmax(np.concatenate([rgpe_1.values.flatten(), rgpe_2.values.flatten()]))
# Common color scale for FFgpe heatmaps
ffgpe_min = np.nanmin(np.concatenate([FF_1.values.flatten(), FF_2.values.flatten()]))
ffgpe_max = np.nanmax(np.concatenate([FF_1.values.flatten(), FF_2.values.flatten()]))
# COmmon color scale for BIgpe heatmaps
bigpe_min = np.nanmin(np.concatenate([BI_1.values.flatten(), BI_2.values.flatten()]))
bigpe_max = np.nanmax(np.concatenate([BI_1.values.flatten(), BI_2.values.flatten()]))

fig = plt.figure(figsize=(10, 12))
gs_outer = GridSpec(4, 1, figure=fig, height_ratios=[1, 1, 1, 1.5], hspace=0.45)
# widen the gap so colorbars don't overlap the last heatmap column
gs_top = gs_outer[0].subgridspec(1, 2, wspace=0.2, width_ratios=[2, 1])
gs_mid = gs_outer[1].subgridspec(1, 2, wspace=0.2, width_ratios=[2, 1])
gs_dow = gs_outer[2].subgridspec(1, 2, wspace=0.2, width_ratios=[2, 1])
gs_bottom = gs_outer[3].subgridspec(1, 3, wspace=0.35, width_ratios=[0.05, 1, 1])

# Row 1: rmsn heatmaps + colorbars
ax_hleft, ax_hright = gs_top[0].subgridspec(1, 3, width_ratios=[1, 1, 0.05]), gs_top[1].subgridspec(1, 2, width_ratios=[1, 0.05])
ax_h1, ax_h2, cb_rate = fig.add_subplot(ax_hleft[0, 0]), fig.add_subplot(ax_hleft[0, 1]), fig.add_subplot(ax_hleft[0, 2])
ax_h3, cb_rate_diff = fig.add_subplot(ax_hright[0, 0]), fig.add_subplot(ax_hright[0, 1])
ax_h1.text(-0.2, 1.2, 'A', transform=ax_h1.transAxes, fontsize=16, va='top')
im_h1 = sns.heatmap(rgpe_1, annot=False, fmt=".2f", cmap='viridis', vmin=rate_min, vmax=rate_max, cbar=False, ax=ax_h1)
ax_h1.set_title(r'$N_{fsi}$' + f'={Nf_1}')
im_h2 = sns.heatmap(rgpe_2, annot=False, fmt=".2f", cmap='viridis', vmin=rate_min, vmax=rate_max, cbar=False, ax=ax_h2)
ax_h2.set_title( r'$N_{fsi}$' + f'={Nf_2}')
im_h3 = sns.heatmap(rgpe_diff, annot=False, fmt=".2f", cmap='coolwarm', cbar=False, ax=ax_h3)
ax_h3.set_title('Difference')
fig.colorbar(im_h1.collections[0], cax=cb_rate, label=r'$r_{gpe}$ (Hz)')
fig.colorbar(im_h3.collections[0], cax=cb_rate_diff, label=r'$\Delta r_{gpe}$ (Hz)')
for ax in [ax_h1]:
    ax.set_xlabel(r'$W_{in}$'); ax.set_ylabel(r'$B_{in}$')
for ax in [ax_h2, ax_h3]:
    ax.set_xlabel(''); ax.set_ylabel(''); ax.set_xticks([]); ax.set_yticks([])

# Row 2: Wmsn heatmaps + colorbars
ax_wleft, ax_wright = gs_mid[0].subgridspec(1, 3, width_ratios=[1, 1, 0.05]), gs_mid[1].subgridspec(1, 2, width_ratios=[1, 0.05])
ax_w1, ax_w2, cb_w = fig.add_subplot(ax_wleft[0, 0]), fig.add_subplot(ax_wleft[0, 1]), fig.add_subplot(ax_wleft[0, 2])
ax_w3, cb_w_diff = fig.add_subplot(ax_wright[0, 0]), fig.add_subplot(ax_wright[0, 1])
ax_w1.text(-0.2, 1.2, 'B', transform=ax_w1.transAxes, fontsize=16, va='top')
im_w1 = sns.heatmap(FF_1, annot=False, fmt=".3f", cmap='viridis', vmin=ffgpe_min, vmax=ffgpe_max, cbar=False, ax=ax_w1)
im_w2 = sns.heatmap(FF_2, annot=False, fmt=".3f", cmap='viridis', vmin=ffgpe_min, vmax=ffgpe_max, cbar=False, ax=ax_w2)
im_w3 = sns.heatmap(FF_diff, annot=False, fmt=".3f", cmap='coolwarm', cbar=False, ax=ax_w3)
fig.colorbar(im_w1.collections[0], cax=cb_w, label=r'$FF_{gpe}$')
fig.colorbar(im_w3.collections[0], cax=cb_w_diff, label=r'$\Delta FF_{gpe}$')
for ax in [ax_w1, ax_w2, ax_w3]:
    ax.set_xlabel(''); ax.set_ylabel(''); ax.set_xticks([]); ax.set_yticks([])

# Row 3: BIgpe heatmaps + colorbars
ax_bleft, ax_bright = gs_dow[0].subgridspec(1, 3, width_ratios=[1, 1, 0.05]), gs_dow[1].subgridspec(1, 2, width_ratios=[1, 0.05])
ax_b1, ax_b2, cb_b = fig.add_subplot(ax_bleft[0, 0]), fig.add_subplot(ax_bleft[0, 1]), fig.add_subplot(ax_bleft[0, 2])
ax_b3, cb_b_diff = fig.add_subplot(ax_bright[0, 0]), fig.add_subplot(ax_bright[0, 1])
ax_b1.text(-0.2, 1.2, 'C', transform=ax_b1.transAxes, fontsize=16, va='top')
im_b1 = sns.heatmap(BI_1, annot=False, fmt=".3f", cmap='viridis', vmin=bigpe_min, vmax=bigpe_max, cbar=False, ax=ax_b1)
im_b2 = sns.heatmap(BI_2, annot=False, fmt=".3f", cmap='viridis', vmin=bigpe_min, vmax=bigpe_max, cbar=False, ax=ax_b2)
im_b3 = sns.heatmap(BI_diff, annot=False, fmt=".3f", cmap='coolwarm', cbar=False, ax=ax_b3)
fig.colorbar(im_b1.collections[0], cax=cb_b, label=r'$BI_{gpe}$')
fig.colorbar(im_b3.collections[0], cax=cb_b_diff, label=r'$\Delta BI_{gpe}$')
for ax in [ax_b1, ax_b2, ax_b3]:
    ax.set_xlabel(''); ax.set_ylabel(''); ax.set_xticks([]); ax.set_yticks([])

# Row 3: scatters (two columns)
ax_s0 = fig.add_subplot(gs_bottom[0, 0])
ax_s0.axis('off')
ax_s1 = fig.add_subplot(gs_bottom[0, 1])
ax_s2 = fig.add_subplot(gs_bottom[0, 2])
ax_s0.text(-0.2, 1.2, 'D', transform=ax_s0.transAxes, fontsize=16, va='top')
scatter_diff(ax_s1, rgpe_diff, FF_diff,  r'$\Delta r_{gpe}$', r'$\Delta FF_{gpe}$', 'Across-trial variablility')
x = rgpe_diff.values.flatten(); y = FF_diff.values.flatten()
m = np.isfinite(x) & np.isfinite(y)
slope, intercept, r_value, p_value, std_err = linregress(x, y)
print(r_value, p_value)
scatter_diff(ax_s2, rgpe_diff, BI_diff,  '', r'$\Delta BI_{gpe}$', 'Bursting index')
plt.setp(ax_s2.get_xticklabels(), visible=False)
x = rgpe_diff.values.flatten(); y = BI_diff.values.flatten()
m = np.isfinite(x) & np.isfinite(y)
slope, intercept, r_value, p_value, std_err = linregress(x, y)
print(r_value, p_value)

plt.savefig(pltpath + '/rc_gpe.eps', dpi=300, bbox_inches='tight')
plt.close()

-0.25034959311050164 0.06525979796857082
-0.04643117772568469 0.736406922957595


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.
